# 45 — Fresh OOT 2026

First post-M4 untouched out-of-time evaluation. The cohort contains only 2026 contracts with a complete 180-day label horizon, therefore it is intentionally partial-year.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

import pandas as pd
for previous in range(40, 45):
    path = paths.reports / "robustness" / f"{previous}_validation.parquet"
    assert path.exists(), f"Falta {path.name}; ejecuta M5 en orden."
    check = pd.read_parquet(path)
    blocking = check.loc[check["blocking"]]
    assert not blocking.empty and blocking["passed"].all(), f"Notebook {previous} no está aprobado."
from argos.robustness.fresh_oot import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-45-fresh-oot-2026-v1'

In [2]:
from argos.robustness.fresh_oot import build_fresh_oot_2026

result = build_fresh_oot_2026(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=False,
)
result.metrics


[45] materializing untouched eligible OOT cohort: 2026-01-01 through 2026-03-13...


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,ARGOS_V1_frozen,fresh_oot_2026,529054,7255,0.013713,0.053274,0.744544,0.06571,0.013238,0.003093


In [3]:
result.ranking


,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain,tie_break
0,ARGOS_V1_frozen,fresh_oot_2026,0.001,530,19,0.035849,0.002619,2.614209,0.002619,score_desc/raw_score_desc/contract_id_asc
1,ARGOS_V1_frozen,fresh_oot_2026,0.005,2646,111,0.041950,0.015300,3.059114,0.015300,score_desc/raw_score_desc/contract_id_asc
2,ARGOS_V1_frozen,fresh_oot_2026,0.010,5291,454,0.085806,0.062578,6.257209,0.062578,score_desc/raw_score_desc/contract_id_asc
3,ARGOS_V1_frozen,fresh_oot_2026,0.025,13227,1170,0.088455,0.161268,6.450407,0.161268,score_desc/raw_score_desc/contract_id_asc
4,ARGOS_V1_frozen,fresh_oot_2026,0.050,26453,2007,0.075870,0.276637,5.532673,0.276637,score_desc/raw_score_desc/contract_id_asc
5,ARGOS_V1_frozen,fresh_oot_2026,0.100,52906,3576,0.067592,0.492901,4.928959,0.492901,score_desc/raw_score_desc/contract_id_asc


In [4]:
result.reliability


,model,split,bin,rows,mean_predicted_probability,observed_rate,min_score,max_score
0,ARGOS_V1_frozen,fresh_oot_2026,0,67104,0.002712,0.009448,0.000000,0.002712
1,ARGOS_V1_frozen,fresh_oot_2026,1,25712,0.002738,0.003228,0.002738,0.002738
2,ARGOS_V1_frozen,fresh_oot_2026,2,23621,0.002831,0.003641,0.002793,0.002831
3,ARGOS_V1_frozen,fresh_oot_2026,3,50181,0.002936,0.003966,0.002936,0.002936
4,ARGOS_V1_frozen,fresh_oot_2026,4,24696,0.003468,0.004657,0.003360,0.003488
5,ARGOS_V1_frozen,fresh_oot_2026,5,31092,0.003658,0.003538,0.003658,0.003658
6,ARGOS_V1_frozen,fresh_oot_2026,6,65738,0.004818,0.007332,0.004230,0.004873
7,ARGOS_V1_frozen,fresh_oot_2026,7,43541,0.005441,0.005351,0.005333,0.005447
8,ARGOS_V1_frozen,fresh_oot_2026,8,32614,0.006379,0.007297,0.006242,0.006448
9,ARGOS_V1_frozen,fresh_oot_2026,9,28756,0.007545,0.006016,0.007005,0.008547


In [5]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,fresh_oot_population_sufficient,holdout,True,True,"{""minimum_positives"": 100, ""minimum_rows"": 200...",fresh OOT cohort has enough fully observed con...
1,fresh_oot_strictly_after_consumed_test,governance,True,True,"{""consumed_test_year"": 2025, ""max_t0"": ""2026-0...",fresh OOT cohort starts strictly after consume...
2,fresh_oot_grain_unique,grain,True,True,"{""distinct_contracts"": 529054, ""predictions"": ...",fresh OOT evaluation has one prediction per co...
3,fresh_oot_uses_exact_frozen_v1,lineage,True,True,"{""calibrator_sha256"": ""95e8e506f92b191f7315a1f...",fresh OOT scoring uses exactly the M4 model/ca...
4,fresh_oot_pr_auc_above_prevalence,usefulness,True,True,"{""pr_auc"": 0.05327382904758357, ""prevalence"": ...",frozen V1 has PR-AUC above the no-skill preval...
5,fresh_oot_lift_operational_k,usefulness,True,True,"{""fraction"": 0.01, ""lift_at_k"": 6.257209223990...",frozen V1 remains useful above random review a...
6,fresh_oot_probabilities_finite,quality,True,True,"{""max"": 0.130938450737187, ""min"": 0.0}",fresh OOT calibrated scores are finite probabi...


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("45 ✅ PASS" if result.ready_for_next_notebook else "45 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Fresh OOT artifact:", result.artifact)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


45 ✅ PASS
Blocking checks: 7/7
Fresh OOT artifact: /Users/macm1pro/Desktop/projects/argos/argos-code/data/m5/fresh_oot_2026.parquet
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_fresh_oot_2026.json
